In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/extensions.py:151: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  mod.load_ipython_extension(self.shell)


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/annotated/checkpoints/pre_cell_10.pickle

trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'SEP_COMMA', 'REGRESSOR']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'SEP_COMMA', 'REGRESSOR']
me:  2
me:  2
me:  2
me:  2
trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['FASTAI_LEARNING_RATE']
me:  2
trying: ['random']
me:  1
trying: ['df']
me:  6
trying: ['factor']
me:  5
trying: ['np']
me:  1


Declaring variable warnings
Declaring variable random
Declaring variable np
Declaring variable shutil
Declaring variable pd
Declaring variable traceback
Declaring variable SEP_DOLLAR
Declaring variable CONVERT_TO_CAT
Declaring variable VARIABLE_FILES
Declaring variable AUTO_ADJUST_LEARNING_RATE
Declaring variable SEP_DOLLAR
Declaring variable CONVERT_TO_CAT
Declaring variable VARIABLE_FILES
Declaring variable AUTO_ADJUST_LEARNING_RATE
Declaring variable SHUFFLE_DATA
Declaring variable REGRESSOR
Declaring variable SEP_COMMA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SHUFFLE_DATA
Declaring variable REGRESSOR
Declaring variable SEP_COMMA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SHUFFLE_DATA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SEP_COMMA
Declaring variable REGRESSOR
Declaring variable SHUFFLE_DATA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SEP_COMMA
Declaring variable REGRESSOR
Declaring variable SEP_DOLLAR
Declaring variable CONV

In [4]:
%%RecordEvent
%%time
### cell 10 ###

import os

target = ''
target_str = ''
targets = []

# Loop through every possible target column (Continuous)
for i in range(len(df.columns) - 1, 0, -1):
    col = df.columns[i]
    try:
        # Convert column to numeric and drop NaNs to match original semantics
        df[col] = pd.to_numeric(df[col], errors='coerce').dropna()
        target = col
        target_str = target.replace('/', '-')
    except:
        continue
    print(f'Target Variable: {target}')

    # Create project config files if they don't exist
    os.makedirs(PARAM_DIR, exist_ok=True)
    for fname in ['cats.txt', 'conts.txt', 'cols_to_delete.txt']:
        fpath = os.path.join(PARAM_DIR, fname)
        if not os.path.exists(fpath):
            with open(fpath, 'w'):
                pass

    # Drop duplicates and optionally shuffle
    df = df.drop_duplicates()
    if SHUFFLE_DATA:
        df = df.sample(frac=1).reset_index(drop=True)

    # Convert any boolean columns to uint8
    for c in df.columns:
        if pd.api.types.is_bool_dtype(df[c]):
            df[c] = df[c].astype('uint8')

    # Remove columns listed in cols_to_delete.txt
    with open(os.path.join(PARAM_DIR, 'cols_to_delete.txt'), 'r') as f:
        cols_to_delete = f.read().splitlines()
    for c in cols_to_delete:
        if c in df.columns:
            try:
                del df[c]
            except:
                pass

    # Fill all remaining NaNs with zero
    try:
        df = df.fillna(0)
    except:
        pass

    # Auto detect categorical vs continuous
    nunique = df.nunique()
    count = df.count()
    cat_mask = nunique / count < 0.05
    cats = cat_mask[cat_mask].index.tolist()
    conts = cat_mask[~cat_mask].index.tolist()

    # Remove target from variable lists
    if target in conts:
        conts.remove(target)
    if target in cats:
        cats.remove(target)

    # Ensure target is numeric
    try:
        df[target] = pd.to_numeric(df[target], errors='coerce').dropna()
    except:
        pass

    # If provided, override cats/conts from files
    if VARIABLE_FILES:
        with open(os.path.join(PARAM_DIR, 'cats.txt'), 'r') as f:
            cats = f.read().splitlines()
        with open(os.path.join(PARAM_DIR, 'conts.txt'), 'r') as f:
            conts = f.read().splitlines()

    # Convert all continuous variables to numeric
    for c in conts:
        try:
            df[c] = pd.to_numeric(df[c], errors='coerce').dropna()
        except:
            print(f'Could not convert {c} to float.')

    # Extra conversions if breakpoint enabled
    if ENABLE_BREAKPOINT:
        cont_list = []
        for cont in conts:
            focus_cont = cont
            cont_list.append(cont)
        for var in cont_list:
            try:
                df[var] = pd.to_numeric(df[var], errors='coerce').dropna()
            except:
                print(f'Could not convert {var} to float.')
                cont_list.remove(var)
                if CONVERT_TO_CAT:
                    cats.append(var)
                pass

Target Variable: Profit_processed


Target Variable: Discount_processed
Target Variable: Quantity_processed
Target Variable: Sales_processed


Target Variable: Sub-Category_processed
Target Variable: Category_processed
Target Variable: Region_processed


Target Variable: Postal Code_processed
Target Variable: State_processed
Target Variable: City_processed


Target Variable: Country_processed
Target Variable: Segment_processed
Target Variable: Ship Mode_processed


Target Variable: Profit
Target Variable: Discount
Target Variable: Quantity


Target Variable: Sales
Target Variable: Sub-Category
Target Variable: Category


Target Variable: Region
Target Variable: Postal Code
Target Variable: State


Target Variable: City
Target Variable: Country
Target Variable: Segment


CPU times: user 2.13 s, sys: 86.8 ms, total: 2.22 s
Wall time: 2.21 s


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten/cell_10/checkpoints/post_cell_10_try_4.pickle

migration speed (bps): 784821422.1724236
---------------------------
variables to migrate:
cat_mask 48
fname 67
df 48
SEP_DOLLAR 24
conts 104
focus_cont 65
SHUFFLE_DATA 28
count 48
REGRESSOR 28
cont 65
SEP_COMMA 28
cont_list 120
np 72
AUTO_ADJUST_LEARNING_RATE 24
target 56
PROJECT_NAME 59
f 208
CONVERT_TO_CAT 24
c 65
VARIABLE_FILES 24
var 65
FASTAI_LEARNING_RATE 24
ENABLE_BREAKPOINT 28
targets 56
i 28
fpath 170
shutil 72
PARAM_DIR 151
pd 72
traceback 72
random 72
warnings 72
target_str 56
cols_to_delete 56
cats 216
col 56
factor 28
nunique 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten/cell_10/checkpoints/post_cell_10_try_4.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['PARAM_DIR', 'df']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['df', 'PARAM_DIR']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables ['df']
Intermediate variables ['factor']
Future variables ['PARAM_DIR']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables ['df', 'col']
Intermediate variables []
Future variables ['PARAM_DIR']
Modified dataframes
  df
    Input columns: set()
    Changed columns: set()
    Created columns: {'Segment_processed', 'Profit_processed', 'Region_processed', 'Category_processed', 'State_processed', 'Ship Mode_processed', 'Quantity_processed', 'Discount_processed', 'City_processed', 'Sub-Category_processed', 'Country_processed', 'Sales_processed', 'Postal Code_processed'}
    Deleted columns: set()
======= Cell 4 =======
In

In [7]:

with open("/home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/opt_cell_exec_info_10_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[10], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
df_opt = df
%LoadCheckpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/annotated/checkpoints/post_cell_10.pickle
assert compare_df(df_opt, df)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
